In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 625.0 kB/s eta 0:00:00


In [ ]:
import pandas as pd
import xlsxwriter

amazon = pd.read_csv("amazon_tv_data.csv")
ebay = pd.read_csv("ebay_tv.csv")
target = pd.read_csv("target_tv_data_clean.csv")
walmart = pd.read_csv("walmart_tvs.csv")

In [ ]:
amazon["store"] = "Amazon"
ebay["store"] = "eBay"
target["store"] = "Target"
walmart["store"] = "Walmart"

df_all = pd.concat([amazon, ebay, target, walmart], ignore_index=True)

df_all["screen_size_inch"] = pd.to_numeric(df_all["screen_size_inch"], errors='coerce')
df_all["price"] = pd.to_numeric(df_all["price"], errors='coerce')

df_all["smart_feature"] = df_all["smart_feature"].fillna("").astype(str)

df_all["Smart_TV"] = df_all["smart_feature"].str.lower().str.contains("yes").map({True: "yes", False: "no"})

def segment_size(size):
    try:
        size = float(size)
        if size < 40:
            return "Small"
        elif 40 <= size <= 55:
            return "Medium"
        else:
            return "Large"
    except:
        return "Unknown"

df_all["size_segment"] = df_all["screen_size_inch"].apply(segment_size)

df_all["price"] = df_all["price"].round(2)

with pd.ExcelWriter("TV_Comparison_total.xlsx", engine="xlsxwriter") as writer:
    workbook = writer.book
    header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1})


    df_all.to_excel(writer, sheet_name="All_TVs", index=False)
    all_ws = writer.sheets["All_TVs"]

    for col_num, value in enumerate(df_all.columns.values):
        all_ws.write(0, col_num, value, header_fmt)


    for store_name, df_store in df_all.groupby("store"):
        worksheet = workbook.add_worksheet(store_name)


        worksheet.write(0, 0, "product_id", header_fmt)
        worksheet.write(0, 1, "product_name", header_fmt)
        worksheet.write(0, 2, "product_brand", header_fmt)
        worksheet.write(0, 3, "category", header_fmt)
        worksheet.write(0, 4, "price", header_fmt)
        worksheet.write(0, 5, "rating", header_fmt)
        worksheet.write(0, 6, "rating_count", header_fmt)
        worksheet.write(0, 7, "product_link", header_fmt)
        worksheet.write(0, 8, "smart_feature", header_fmt)
        worksheet.write(0, 9, "screen_size_inch", header_fmt)
        worksheet.write(0, 10, "size_segment", header_fmt)
        worksheet.write(0, 11, "Smart_TV", header_fmt)
        worksheet.write(0, 12, "store", header_fmt)


        for row_num, row in enumerate(df_store.itertuples(index=False), start=1):
            worksheet.write(row_num, 0, row.product_id)
            worksheet.write(row_num, 1, row.product_name)
            worksheet.write(row_num, 2, row.product_brand)
            worksheet.write(row_num, 3, row.category)
            worksheet.write(row_num, 4, row.price)
            worksheet.write(row_num, 5, row.rating)
            worksheet.write(row_num, 6, row.rating_count)
            worksheet.write(row_num, 7, row.product_link)
            worksheet.write(row_num, 8, row.smart_feature)
            worksheet.write(row_num, 9, row.screen_size_inch)
            worksheet.write(row_num, 10, row.size_segment)
            worksheet.write(row_num, 11, row.Smart_TV)
            worksheet.write(row_num, 12, row.store)


    summary_sheet = workbook.add_worksheet("Summary")
    summary_sheet.write(0, 0, "Store", header_fmt)
    summary_sheet.write(0, 1, "Average Price", header_fmt)
    summary_sheet.write(0, 2, "Average Screen Size", header_fmt)
    summary_sheet.write(0, 3, "Number of Smart TVs", header_fmt)
    summary_sheet.write(0, 4, "Total TVs", header_fmt)

    stores = ["Amazon", "eBay", "Target", "Walmart"]
    for row, store in enumerate(stores, start=1):
        summary_sheet.write(row, 0, store)


